# Genomic VEP — Training Notebook

Fine-tunes Nucleotide Transformer v2 (50M) on ClinVar data to predict variant pathogenicity.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train.parquet`, `val.parquet`, `test.parquet` to the Files panel (left sidebar)
3. Run all cells

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers datasets scikit-learn

## 2. Verify GPU

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## 3. Upload data

Upload your parquet files to `/content/data/`. You can either:
- Drag and drop into the Files panel, or
- Run the cell below to upload via Google Drive

In [ ]:
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # select train.parquet, val.parquet, test.parquet
# for name, data in uploaded.items():
#     with open(os.path.join(DATA_DIR, name), "wb") as f:
#         f.write(data)

# Option B: Google Drive (if you uploaded parquets there)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/genomic-vep/data/*.parquet /content/data/

# Verify files exist
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}.parquet")
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {split}.parquet: {size_mb:.1f} MB")
    else:
        print(f"  MISSING: {path}")

## 4. Dataset & DataLoader

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"


class VariantDataset(Dataset):
    def __init__(self, split_type: str, tokenizer, max_length: int = 256) -> None:
        self.df = pd.read_parquet(os.path.join(DATA_DIR, f"{split_type}.parquet"))
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"Loaded {split_type}: {len(self.df)} samples")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        ref_tokens = self.tokenizer(
            row["ref_seq"], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        alt_tokens = self.tokenizer(
            row["alt_seq"], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "ref_ids": ref_tokens["input_ids"].squeeze(0),
            "ref_mask": ref_tokens["attention_mask"].squeeze(0),
            "alt_ids": alt_tokens["input_ids"].squeeze(0),
            "alt_mask": alt_tokens["attention_mask"].squeeze(0),
            "label": torch.tensor(row["label"], dtype=torch.float32),
        }


print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f"Tokenizer loaded (vocab size: {tokenizer.vocab_size})")

BATCH_SIZE = 32
train_ds = VariantDataset("train", tokenizer)
val_ds = VariantDataset("val", tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain: {len(train_loader)} batches, Val: {len(val_loader)} batches")

## 5. Model

In [ ]:
import torch.nn as nn
from transformers import AutoModel


class VariantClassifier(nn.Module):
    def __init__(self, dropout: float = 0.1) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def get_embedding(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)

    def forward(
        self, ref_ids: torch.Tensor, ref_mask: torch.Tensor, alt_ids: torch.Tensor, alt_mask: torch.Tensor
    ) -> torch.Tensor:
        ref_emb = self.get_embedding(ref_ids, ref_mask)
        alt_emb = self.get_embedding(alt_ids, alt_mask)
        diff = alt_emb - ref_emb
        return self.head(diff).squeeze(-1)


print("Loading NT-v2 model...")
model = VariantClassifier().to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,} | Frozen: {total - trainable:,}")

## 6. Training

In [ ]:
from sklearn.metrics import roc_auc_score

EPOCHS = 5
LR = 1e-3
POS_WEIGHT = 2.0  # upweight pathogenic class (~31% of data)

optimizer = torch.optim.Adam(model.head.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))


def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch["ref_ids"].to(device), batch["ref_mask"].to(device),
                batch["alt_ids"].to(device), batch["alt_mask"].to(device),
            )
            all_labels.extend(batch["label"].tolist())
            all_probs.extend(torch.sigmoid(logits).cpu().tolist())
    return roc_auc_score(all_labels, all_probs)


best_auroc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        ref_ids = batch["ref_ids"].to(device)
        ref_mask = batch["ref_mask"].to(device)
        alt_ids = batch["alt_ids"].to(device)
        alt_mask = batch["alt_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(ref_ids, ref_mask, alt_ids, alt_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if num_batches % 200 == 0:
            print(f"  Epoch {epoch} | Batch {num_batches}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / num_batches
    val_auroc = evaluate(model, val_loader)

    print(f"Epoch {epoch}/{EPOCHS} | Train loss: {avg_loss:.4f} | Val AUROC: {val_auroc:.4f}")

    if val_auroc > best_auroc:
        best_auroc = val_auroc
        torch.save(model.head.state_dict(), "/content/best_model.pth")
        print(f"  Saved best model (AUROC: {best_auroc:.4f})")

print(f"\nTraining complete. Best Val AUROC: {best_auroc:.4f}")

## 7. Test evaluation

In [ ]:
# Load best checkpoint and evaluate on test set
model.head.load_state_dict(torch.load("/content/best_model.pth"))

test_ds = VariantDataset("test", tokenizer)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

test_auroc = evaluate(model, test_loader)
print(f"Test AUROC: {test_auroc:.4f}")

## 8. Download checkpoint

In [ ]:
from google.colab import files
files.download("/content/best_model.pth")